In [2]:
# https://raw.githubusercontent.com/nkmwicz/worldcup2018data/
# refs/heads/main/cleaned_events_world_cup2018.csv

import pandas as pd, scipy.stats as stats

url = "https://raw.githubusercontent.com/nkmwicz/worldcup2018data/refs/heads/main/cleaned_events_world_cup2018.csv"
df = pd.read_csv(url)
df.head(1)

,eventId,subEventName,tags,playerId,matchId,eventName,teamId,matchPeriod,eventSec,subEventId,id,x1,y1,x2,y2
0,8,Simple pass,['Accurate'],Mohammad Ibrahim Al Sahlawi,"Russia - Saudi Arabia, 5 - 0",Pass,Saudi Arabia,1H,1.656214,Simple pass,258612104,50,50,35.0,53.0


In [3]:
len(df)

101759

In [4]:
df["eventName"].unique()

<StringArray>
[                   'Pass',                    'Duel',
               'Free Kick',                    'Foul',
      'Others on the ball',                    'Shot',
            'Save attempt',                 'Offside',
 'Goalkeeper leaving line']
Length: 9, dtype: str

In [5]:
# is each event a pass?
# is each event a goal?
# is each event a shot?
df["pass"] = (df["eventName"] == "Pass").astype(int)
df["shot"] = (df["eventName"] == "Shot").astype(int)
df["shot"].head()

0    0
1    0
2    0
3    0
4    0
Name: shot, dtype: int64

In [6]:
df["subEventName"].unique()

<StringArray>
[            'Simple pass',               'High pass',
                'Air duel',   'Ground attacking duel',
   'Ground defending duel',                'Throw in',
                    'Foul',               'Free Kick',
               'Clearance',                   'Touch',
  'Ground loose ball duel',                  'Corner',
               'Head pass',                  'Launch',
            'Acceleration',                    'Shot',
              'Smart pass',                   'Cross',
                'Reflexes',                       nan,
               'Hand pass',               'Goal kick',
         'Free kick cross',               'Hand foul',
            'Save attempt',          'Free kick shot',
 'Goalkeeper leaving line',                 'Penalty',
          'Late card foul',            'Violent Foul',
                 'Protest',        'Out of game foul',
          'Time lost foul',              'Simulation']
Length: 34, dtype: str

In [8]:
import ast 
df["tags"] = df["tags"].apply(ast.literal_eval)

In [12]:
tags = set([tag for taglist in df["tags"] for tag in taglist])
tags

{'Accurate',
 'Anticipated',
 'Anticipation',
 'Assist',
 'Blocked',
 'Counter attack',
 'Dangerous ball lost',
 'Direct',
 'Fairplay',
 'Feint',
 'Free space left',
 'Free space right',
 'Goal',
 'Head/body',
 'High',
 'Indirect',
 'Interception',
 'Key pass',
 'Left foot',
 'Lost',
 'Missed ball',
 'Neutral',
 'Not accurate',
 'Opportunity',
 'Own goal',
 'Position: Goal center',
 'Position: Goal center left',
 'Position: Goal center right',
 'Position: Goal high center',
 'Position: Goal high left',
 'Position: Goal high right',
 'Position: Goal low center',
 'Position: Goal low left',
 'Position: Goal low right',
 'Position: Out center left',
 'Position: Out center right',
 'Position: Out high center',
 'Position: Out high left',
 'Position: Out high right',
 'Position: Out low left',
 'Position: Out low right',
 'Position: Post center left',
 'Position: Post center right',
 'Position: Post high center',
 'Position: Post high left',
 'Position: Post high right',
 'Position: Post lo

In [13]:
# def get_goals(taglist):
#     return "Goal" in taglist

df["goal"] = (
    (df["eventName"] == "Shot") & 
    (df["tags"].apply(lambda taglist: "Goal" in taglist))
    ).astype(int)

# df["goal"] = df.apply(lambda row: (row["eventName"] == "Shot") & ("Goal" in row["tags"]) , axis=1).astype(int)

In [14]:
df.head(1)

,eventId,subEventName,tags,playerId,matchId,eventName,teamId,matchPeriod,eventSec,subEventId,id,x1,y1,x2,y2,pass,shot,goal
0,8,Simple pass,[Accurate],Mohammad Ibrahim Al Sahlawi,"Russia - Saudi Arabia, 5 - 0",Pass,Saudi Arabia,1H,1.656214,Simple pass,258612104,50,50,35.0,53.0,1,0,0


In [18]:
from typing import List

def get_stats(columns: List[str], normalized:bool=False):
    """
    Args: 
        columns (List[str]): columns to groupby 
        normalized (bool): should we normalize shots, passes and goals to game?
    Returns:
        None
    """
    grp = df.groupby(columns).agg(
        passes=("pass", "sum"),
        shots=("shot", "sum"),
        goals=("goal", "sum"),
        matches=("matchId", "nunique")
    ).reset_index()
    if normalized:
        grp["passes"] = grp["passes"] / grp["matches"]
        grp["shots"] = grp["shots"] / grp["matches"]
        grp["goals"] = grp["goals"] / grp["matches"]
    
    r,p = stats.pearsonr(grp["passes"], grp["shots"])
    r1,p1 = stats.pearsonr(grp["passes"], grp["goals"])
    
    print(f"agg by {'-'.join(columns)}{"norm" if normalized else None}: shots r:{r:.4f} p:{p:.4f} r2:{r**2:.4f}")
    print(f"agg by {'-'.join(columns)}{"norm" if normalized else None}: goals r:{r1:.4f} p:{p1:.4f} r2:{r1**2:.4f}")

## By Matches

In [19]:
get_stats(["matchId"])

agg by matchIdNone: shots r:0.1996 p:0.1138 r2:0.0398
agg by matchIdNone: goals r:0.0262 p:0.8369 r2:0.0007


## By Teams Total

In [20]:
get_stats(["teamId"])

agg by teamIdNone: shots r:0.8932 p:0.0000 r2:0.7979
agg by teamIdNone: goals r:0.8541 p:0.0000 r2:0.7295


## By Teams Normalized

In [21]:
get_stats(["teamId"], normalized=True)

agg by teamIdnorm: shots r:0.6667 p:0.0000 r2:0.4445
agg by teamIdnorm: goals r:0.4271 p:0.0148 r2:0.1824


## By Match & Team

In [22]:
get_stats(["matchId","teamId"])

agg by matchId-teamIdNone: shots r:0.5642 p:0.0000 r2:0.3184
agg by matchId-teamIdNone: goals r:0.1200 p:0.1772 r2:0.0144


In [23]:
grp = df.groupby(["matchId", "teamId"]).agg(
        passes=("pass", "sum"),
        shots=("shot", "sum"),
        goals=("goal", "sum"),
        matches=("matchId", "nunique")
    ).reset_index()
grp

,matchId,teamId,passes,shots,goals,matches
0,"Argentina - Croatia, 0 - 3",Argentina,476,10,0,1
1,"Argentina - Croatia, 0 - 3",Croatia,334,13,3,1
2,"Argentina - Iceland, 1 - 1",Argentina,744,22,1,1
3,"Argentina - Iceland, 1 - 1",Iceland,183,8,1,1
4,"Australia - Peru, 0 - 2",Australia,512,11,0,1
...,...,...,...,...,...,...
123,"Uruguay - Portugal, 2 - 1",Uruguay,238,6,2,1
124,"Uruguay - Russia, 3 - 0",Russia,327,3,0,1
125,"Uruguay - Russia, 3 - 0",Uruguay,466,12,1,1
126,"Uruguay - Saudi Arabia, 1 - 0",Saudi Arabia,553,7,0,1
